## 01- Transformer Classifier Baseline
In this notebook, we will do a baseline using a mix of classicial and neural method (Transformer-based sentence embeddings and logistic regression classifier)

In [1]:
# !pip install sentence-transformers

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    classification_report,
    mean_squared_error,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import cross_val_score

/Users/la/miniconda3/envs/mds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_fname = "../data/splits/train.csv"
dev_fname = "../data/splits/dev.csv"

train_df = pd.read_csv(train_fname)
dev_df = pd.read_csv(dev_fname)

print("Train len =", len(train_df))
print("Dev df =", len(dev_df))

Train len = 132
Dev df = 34


In [4]:
train_df.head()

,text,target,mmse
0,Reaching for a cookie chair possibly moving to...,CN,28
1,As a picture little girl and a little boy stan...,AD,18
2,Tell me everything that you see happening. Eve...,CN,30
3,I think what I want you to do is take a look a...,CN,30
4,Tell me all the action that you can say on wha...,AD,20


In [5]:
dev_df.head()

,text,target,mmse
0,"Everything you see going on in this picture, g...",CN,28
1,"Now you just look at the picture. Okay, this i...",CN,30
2,Tell me everything that you see going on in th...,AD,20
3,Girls and boys getting her one that stores sto...,AD,20
4,"All right, we have a few more tasks to go thro...",CN,30


### Pre-processing
We will lowercase the text for better efficiency

In [6]:
train_df["text_lower"] = train_df["text"].apply(lambda text: text.lower().strip())
dev_df["text_lower"] = dev_df["text"].apply(lambda text: text.lower().strip())

### Apply Sentence Transformer

We can use Sentence Transformer to convert text to a fixed-length embedding vector

In [7]:
vectorizer = SentenceTransformer("sentence-t5-large")  # works best

In [8]:
X_train = vectorizer.encode(train_df["text_lower"])
X_dev = vectorizer.encode(dev_df["text_lower"])

In [9]:
print(X_train.shape)

(132, 768)


In [10]:
print(X_dev.shape)

(34, 768)


### Train and Evaluate MLP Classifier (AD vs CN)

We can now train a multi layer perceptron classifier using processed X_train

In [11]:
def train_eval_model(model, X_train, y_train, X_dev, y_dev, cv, is_classifier=True):
    print("Training neural model...")
    model.fit(X_train, y_train)
    print("Completed")

    y_pred = model.predict(X_train)
    if is_classifier:
        print(classification_report(y_train, y_pred))
    else:
        train_rmse = np.sqrt(mean_squared_error(y_train, y_pred))
        print("Train RMSE: ", train_rmse)

    print("Performing cross-validation...")
    scores = cross_val_score(model, X_train, y_train, cv=cv)
    print("Average =", np.mean(scores))

    y_pred = model.predict(X_dev)
    if is_classifier:
        print(classification_report(y_dev, y_pred))
    else:
        dev_rmse = np.sqrt(mean_squared_error(y_dev, y_pred))
        print("Dev RMSE: ", dev_rmse)

In [12]:
solver = "lbfgs"
alpha = 1e-5
learning_rate_init = 0.0001
hidden_size = 500
cv = 10
max_iter = 500
random_state = 42

In [13]:
y_train = train_df["target"]
y_dev = dev_df["target"]
clf = MLPClassifier(
    solver=solver,
    alpha=alpha,
    hidden_layer_sizes=(hidden_size, hidden_size, hidden_size),
    learning_rate_init=learning_rate_init,
    max_iter=max_iter,
    random_state=random_state,
)

train_eval_model(clf, X_train, y_train, X_dev, y_dev, cv)

Training neural model...
Completed
              precision    recall  f1-score   support

          AD       1.00      1.00      1.00        66
          CN       1.00      1.00      1.00        66

    accuracy                           1.00       132
   macro avg       1.00      1.00      1.00       132
weighted avg       1.00      1.00      1.00       132

Performing cross-validation...
Average = 0.7274725274725274
              precision    recall  f1-score   support

          AD       0.82      0.86      0.84        21
          CN       0.75      0.69      0.72        13

    accuracy                           0.79        34
   macro avg       0.78      0.77      0.78        34
weighted avg       0.79      0.79      0.79        34



### Train and Evaluate MLP Regressor (MMSE scores)

We now continue to train a multi layer perceptron regressor using processed X_train

In [14]:
y_train = train_df["mmse"]
y_dev = dev_df["mmse"]
clf = MLPRegressor(
    solver=solver,
    alpha=alpha,
    hidden_layer_sizes=(hidden_size, hidden_size, hidden_size),
    learning_rate_init=learning_rate_init,
    max_iter=max_iter,
    random_state=random_state,
)
train_eval_model(clf, X_train, y_train, X_dev, y_dev, cv, is_classifier=False)

Training neural model...


/Users/la/miniconda3/envs/mds/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


Completed
Train RMSE:  0.6372642466626149
Performing cross-validation...


/Users/la/miniconda3/envs/mds/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Users/la/miniconda3/envs/mds/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
/Users/la/miniconda3/envs/mds/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs faile

Average = 0.061285453409475096
Dev RMSE:  6.88826712988294
